In [6]:
# =============================================================================
# OralVision M4
# CORRECTED / LEAKAGE-AWARE DATA SPLITTING PIPELINE
# =============================================================================
#
# PURPOSE
# -------
# Correct the original pipeline in which augmentation/derived images could be
# generated BEFORE train/validation/test splitting.
#
# Correct order:
#
#   ORIGINAL SOURCE IMAGES
#           |
#           v
#   SOURCE-IMAGE / PATIENT GROUP IDENTIFICATION
#           |
#           v
#   TRAIN / VALIDATION / TEST SPLIT
#           |
#           +-----------------------------+
#           |                             |
#           v                             v
#        TRAIN                       VAL / TEST
#           |                             |
#           v                             v
#     augmentation                 ORIGINAL ONLY
#     Ben Graham                   UNTOUCHED
#           |
#           v
#       TRAIN ONLY
#
# IMPORTANT
# ---------
# This script provides SOURCE-IMAGE-LEVEL separation.
#
# It does NOT establish patient-level separation unless a genuine patient,
# visit, case, or examination identifier is available.
#
# If the diagnostic section finds evidence that filenames encode patient IDs,
# DO NOT continue blindly. Inspect and modify make_group_id() accordingly.
#
# OUTPUT
# ------
# OUTPUT_DIR/
#
#   train/
#       class_1/
#       ...
#
#   val/
#       class_1/
#       ...
#
#   test/
#       class_1/
#       ...
#
#   split_manifest.csv
#   split_summary.txt
#   duplicate_hash_report.csv
#   class_distribution.csv
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import re
import csv
import shutil
import random
import hashlib
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
from PIL import Image, ImageOps, ImageFilter
from sklearn.model_selection import train_test_split


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# ---------------------------------------------------------------------------
# INPUT
# ---------------------------------------------------------------------------
# IMPORTANT:
# This directory MUST contain the ORIGINAL, pre-augmentation images.
#
# Expected structure:
#
# oral_merged/
#     Calculus/
#         image001.jpg
#         image002.jpg
#     Caries/
#         image003.jpg
#     Gingivitis/
#     Hypodontia/
#     Mouth_Ulcer/
#     Tooth_Discoloration/
#
# Change this path to your actual folder.
# ---------------------------------------------------------------------------

ORIGINAL_DIR = r"C:\Users\R Y Z E N\Desktop\oral_merged"


# ---------------------------------------------------------------------------
# OUTPUT
# ---------------------------------------------------------------------------

OUTPUT_DIR = (
    r"C:\Users\R Y Z E N\Desktop\oral_dataset_corrected_split"
)


# ---------------------------------------------------------------------------
# RANDOM SEED
# ---------------------------------------------------------------------------

SEED = 42


# ---------------------------------------------------------------------------
# SPLIT PROPORTIONS
# ---------------------------------------------------------------------------
#
# 70% train
# 15% validation
# 15% test
#
# These are applied BEFORE augmentation.
# ---------------------------------------------------------------------------

TEST_SPLIT = 0.15
VAL_SPLIT = 0.15


# ---------------------------------------------------------------------------
# AUGMENTATION
# ---------------------------------------------------------------------------
#
# IMPORTANT:
# Keep these settings identical to the original experiment if the objective
# is to isolate the effect of correcting the splitting protocol.
#
# One augmented image is generated per original training image.
# ---------------------------------------------------------------------------

AUGMENTATIONS_PER_IMAGE = 1

ROTATION_RANGE_DEG = 15

HORIZONTAL_FLIP_PROBABILITY = 0.50


# ---------------------------------------------------------------------------
# BEN GRAHAM
# ---------------------------------------------------------------------------
#
# If the original experiment used a different implementation, replace this
# function with the EXACT original implementation.
#
# Do not silently change preprocessing between the old and corrected
# experiments.
# ---------------------------------------------------------------------------

BEN_GRAHAM_SCALE = 300
BEN_GRAHAM_ALPHA = 4.0
BEN_GRAHAM_BETA = 128.0


# ---------------------------------------------------------------------------
# IMAGE EXTENSIONS
# ---------------------------------------------------------------------------

VALID_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
)


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================

random.seed(SEED)
np.random.seed(SEED)


# =============================================================================
# 3. LOGGING
# =============================================================================

def log(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


# =============================================================================
# 4. DIRECTORY CHECKS
# =============================================================================

if not os.path.isdir(ORIGINAL_DIR):
    raise FileNotFoundError(
        f"\nINPUT DIRECTORY NOT FOUND:\n{ORIGINAL_DIR}\n\n"
        "Please change ORIGINAL_DIR to the folder containing the ORIGINAL "
        "pre-augmentation images."
    )


os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 5. LOAD ORIGINAL DATASET
# =============================================================================

def build_samples(data_dir):

    classes = sorted(
        [
            d
            for d in os.listdir(data_dir)
            if os.path.isdir(os.path.join(data_dir, d))
        ]
    )

    if len(classes) < 2:
        raise RuntimeError(
            "Fewer than two class folders were found. "
            "Check ORIGINAL_DIR."
        )

    samples = []

    for cls in classes:

        class_dir = os.path.join(data_dir, cls)

        for fname in sorted(os.listdir(class_dir)):

            if fname.lower().endswith(VALID_EXTENSIONS):

                full_path = os.path.join(class_dir, fname)

                samples.append(
                    {
                        "path": full_path,
                        "class": cls,
                        "filename": fname,
                    }
                )

    return samples, classes


log("=" * 80)
log("ORALVISION M4 — CORRECTED SPLITTING PIPELINE")
log("=" * 80)

log("Loading ORIGINAL source images...")
log(f"Input directory: {ORIGINAL_DIR}")

samples, classes = build_samples(ORIGINAL_DIR)

log(f"Classes found: {classes}")
log(f"Total original images: {len(samples)}")


if len(samples) == 0:
    raise RuntimeError(
        "No image files were found."
    )


# =============================================================================
# 6. ORIGINAL CLASS DISTRIBUTION
# =============================================================================

original_class_counts = Counter(
    sample["class"]
    for sample in samples
)

log("\nOriginal class distribution:")

for cls in classes:
    log(
        f"  {cls:<25} "
        f"{original_class_counts[cls]:>7}"
    )


# =============================================================================
# 7. DIAGNOSTIC: FILENAME STRUCTURE
# =============================================================================
#
# This section DOES NOT claim to discover patient IDs automatically.
#
# It only searches for patterns that deserve human inspection.
# =============================================================================

log("\n" + "=" * 80)
log("DIAGNOSTIC: FILENAME STRUCTURE")
log("=" * 80)

by_class = defaultdict(list)

for sample in samples:
    by_class[sample["class"]].append(
        sample["filename"]
    )


for cls in classes:

    log(f"\n{cls} — first 10 filenames:")

    for fname in sorted(by_class[cls])[:10]:

        log(f"    {fname}")


# =============================================================================
# 8. HEURISTIC PREFIX ANALYSIS
# =============================================================================
#
# Examples:
#
# patient001_1.jpg
# patient001_2.jpg
#
# or
#
# case_123_front.jpg
# case_123_side.jpg
#
# This is ONLY a diagnostic aid.
# =============================================================================

def remove_extension(filename):

    return os.path.splitext(filename)[0]


def heuristic_prefix(filename):

    stem = remove_extension(filename)

    # Remove common trailing numeric counters.
    #
    # Examples:
    # image_001 -> image
    # patient123_1 -> patient123
    #
    prefix = re.sub(
        r"([_\-\s]?\d+)$",
        "",
        stem
    )

    return prefix


prefix_counts = Counter()

for sample in samples:

    prefix = heuristic_prefix(
        sample["filename"]
    )

    prefix_counts[prefix] += 1


repeated_prefixes = {
    prefix: count
    for prefix, count in prefix_counts.items()
    if count > 1
}


log(
    f"\nDistinct heuristic prefixes: "
    f"{len(prefix_counts)}"
)

log(
    f"Prefixes occurring more than once: "
    f"{len(repeated_prefixes)}"
)


if repeated_prefixes:

    log(
        "\nWARNING:"
    )

    log(
        "Repeated filename prefixes were detected."
    )

    log(
        "These may or may not represent patient/case/visit identifiers."
    )

    log(
        "Inspect the examples below BEFORE using the generated dataset."
    )

    for prefix, count in sorted(
        repeated_prefixes.items(),
        key=lambda x: -x[1]
    )[:25]:

        log(
            f"    '{prefix}' -> {count} files"
        )

else:

    log(
        "\nNo repeated heuristic filename prefixes detected."
    )


log(
    "\nIMPORTANT:"
)

log(
    "Automatic filename heuristics cannot establish patient identity."
)

log(
    "If patient/visit identifiers are known outside filenames, "
    "they must be incorporated manually."
)


# =============================================================================
# 9. SOURCE-IMAGE GROUP IDENTIFICATION
# =============================================================================
#
# Current assumption:
#
#   Each original photograph = one source-image group.
#
# Therefore each original file receives a unique group ID.
#
# This prevents augmented derivatives of one source photograph from being
# distributed across train/validation/test because augmentation is performed
# AFTER the split.
#
# If a true patient/case ID becomes available, replace make_group_id().
# =============================================================================

def make_group_id(sample):

    cls = sample["class"]

    fname = sample["filename"]

    # Use normalized absolute path to avoid collisions when two class folders
    # contain identically named images.
    #
    # We deliberately DO NOT use augmentation-derived filenames here because
    # this dataset is built from ORIGINAL source images.
    #
    return f"{cls}::{fname}"


for sample in samples:

    sample["group_id"] = make_group_id(sample)


# =============================================================================
# 10. VERIFY GROUP UNIQUENESS
# =============================================================================

group_ids = [
    sample["group_id"]
    for sample in samples
]

if len(group_ids) != len(set(group_ids)):

    raise RuntimeError(
        "Duplicate source group IDs detected. "
        "The source-image grouping logic must be fixed before continuing."
    )


log(
    "\nSource-image group count: "
    f"{len(group_ids)}"
)


# =============================================================================
# 11. BUILD GROUP-LEVEL LABEL MAP
# =============================================================================

group_to_label = {
    sample["group_id"]: sample["class"]
    for sample in samples
}


unique_groups = sorted(
    group_to_label.keys()
)


unique_group_labels = [
    group_to_label[group_id]
    for group_id in unique_groups
]


# =============================================================================
# 12. TRAIN / VALIDATION / TEST SPLIT
# =============================================================================
#
# IMPORTANT:
#
# NOTHING HAS BEEN AUGMENTED YET.
#
# This is the critical correction.
# =============================================================================

log("\n" + "=" * 80)
log("PERFORMING GROUP-LEVEL SPLIT BEFORE AUGMENTATION")
log("=" * 80)


# First:
#
#   85% train+validation
#   15% test
#

trainval_groups, test_groups = train_test_split(
    unique_groups,
    test_size=TEST_SPLIT,
    stratify=unique_group_labels,
    random_state=SEED,
)


# Convert 15% validation of total data into the corresponding proportion
# inside the 85% train+validation subset.
#
# 0.15 / 0.85 = approximately 0.17647
#

val_fraction_within_trainval = (
    VAL_SPLIT / (1.0 - TEST_SPLIT)
)


trainval_labels = [
    group_to_label[group_id]
    for group_id in trainval_groups
]


train_groups, val_groups = train_test_split(
    trainval_groups,
    test_size=val_fraction_within_trainval,
    stratify=trainval_labels,
    random_state=SEED,
)


train_set = set(train_groups)
val_set = set(val_groups)
test_set = set(test_groups)


# =============================================================================
# 13. HARD LEAKAGE CHECK
# =============================================================================

log("\nRunning split leakage checks...")


if train_set & val_set:

    raise RuntimeError(
        "LEAKAGE DETECTED: train and validation groups overlap."
    )


if train_set & test_set:

    raise RuntimeError(
        "LEAKAGE DETECTED: train and test groups overlap."
    )


if val_set & test_set:

    raise RuntimeError(
        "LEAKAGE DETECTED: validation and test groups overlap."
    )


all_split_groups = (
    train_set
    | val_set
    | test_set
)


if all_split_groups != set(unique_groups):

    missing = (
        set(unique_groups)
        - all_split_groups
    )

    extra = (
        all_split_groups
        - set(unique_groups)
    )

    raise RuntimeError(
        "Split assignment is incomplete or inconsistent.\n"
        f"Missing groups: {len(missing)}\n"
        f"Unexpected groups: {len(extra)}"
    )


log(
    "✓ TRAIN / VALIDATION / TEST GROUPS ARE DISJOINT."
)

log(
    f"Train source images: {len(train_set)}"
)

log(
    f"Validation source images: {len(val_set)}"
)

log(
    f"Test source images: {len(test_set)}"
)


# =============================================================================
# 14. MAP EACH SOURCE IMAGE TO ITS SPLIT
# =============================================================================

split_of = {}

for group_id in train_set:
    split_of[group_id] = "train"

for group_id in val_set:
    split_of[group_id] = "val"

for group_id in test_set:
    split_of[group_id] = "test"


for sample in samples:

    sample["split"] = split_of[
        sample["group_id"]
    ]


# =============================================================================
# 15. CLASS DISTRIBUTION BY SPLIT
# =============================================================================

split_class_counts = {
    "train": Counter(),
    "val": Counter(),
    "test": Counter(),
}


for sample in samples:

    split_class_counts[
        sample["split"]
    ][sample["class"]] += 1


log("\n" + "=" * 80)
log("SOURCE-IMAGE CLASS DISTRIBUTION")
log("=" * 80)


for cls in classes:

    log(
        f"{cls:<25} "
        f"train={split_class_counts['train'][cls]:>6}  "
        f"val={split_class_counts['val'][cls]:>6}  "
        f"test={split_class_counts['test'][cls]:>6}"
    )


# =============================================================================
# 16. CREATE OUTPUT DIRECTORIES
# =============================================================================

log("\nCreating output directories...")


for split in (
    "train",
    "val",
    "test",
):

    for cls in classes:

        os.makedirs(
            os.path.join(
                OUTPUT_DIR,
                split,
                cls,
            ),
            exist_ok=True,
        )


# =============================================================================
# 17. IMAGE HASH FUNCTION
# =============================================================================
#
# Used to detect exact duplicate source files.
# =============================================================================

def file_md5(path):

    md5 = hashlib.md5()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


# =============================================================================
# 18. SOURCE DUPLICATE CHECK
# =============================================================================

log("\nChecking exact duplicate source files...")


hash_to_files = defaultdict(list)


for sample in samples:

    try:

        h = file_md5(
            sample["path"]
        )

        sample["source_md5"] = h

        hash_to_files[h].append(
            sample
        )

    except Exception as e:

        log(
            f"WARNING: Could not hash "
            f"{sample['path']}: {e}"
        )

        sample["source_md5"] = ""


duplicate_hash_rows = []


for h, grouped_samples in hash_to_files.items():

    if len(grouped_samples) > 1:

        for sample in grouped_samples:

            duplicate_hash_rows.append(
                {
                    "md5": h,
                    "class": sample["class"],
                    "filename": sample["filename"],
                    "split": sample["split"],
                    "path": sample["path"],
                }
            )


duplicate_report_path = os.path.join(
    OUTPUT_DIR,
    "duplicate_hash_report.csv"
)


with open(
    duplicate_report_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "md5",
            "class",
            "filename",
            "split",
            "path",
        ],
    )

    writer.writeheader()

    writer.writerows(
        duplicate_hash_rows
    )


if duplicate_hash_rows:

    log(
        "\nWARNING:"
    )

    log(
        f"{len(duplicate_hash_rows)} files belong to "
        "duplicate MD5 groups."
    )

    log(
        "Inspect duplicate_hash_report.csv."
    )

else:

    log(
        "✓ No exact duplicate source files detected."
    )


# =============================================================================
# 19. IMAGE SAVE FUNCTION
# =============================================================================

def save_image(
    img,
    destination_path,
):

    # JPEG output if the extension is JPEG/JPG.
    #
    # PNG/BMP/TIFF are saved using their corresponding extension.
    #
    ext = os.path.splitext(
        destination_path
    )[1].lower()

    if ext in (".jpg", ".jpeg"):

        img.save(
            destination_path,
            quality=95,
            optimize=True,
        )

    else:

        img.save(
            destination_path
        )


# =============================================================================
# 20. AUGMENTATION
# =============================================================================

def augment_image(img):

    """
    Training-only geometric augmentation.

    Matches the intended original design:
        - horizontal flip
        - small random rotation

    IMPORTANT:
    If the original experiment used cropping or another transformation,
    reproduce it here EXACTLY rather than silently changing the protocol.
    """

    aug = img.copy()

    # Horizontal flip
    if random.random() < HORIZONTAL_FLIP_PROBABILITY:

        aug = ImageOps.mirror(
            aug
        )

    # Random rotation
    angle = random.uniform(
        -ROTATION_RANGE_DEG,
        ROTATION_RANGE_DEG,
    )

    aug = aug.rotate(
        angle,
        expand=False,
        fillcolor=(0, 0, 0),
    )

    return aug


# =============================================================================
# 21. BEN GRAHAM NORMALIZATION
# =============================================================================

def ben_graham(
    img,
    scale=BEN_GRAHAM_SCALE,
):

    """
    Ben Graham-style local illumination normalization.

    Formula:

        result = alpha * image
                 - alpha * blurred_image
                 + beta

    IMPORTANT:
    Verify that this implementation exactly matches the implementation used
    for the original OralVision-M4 experiment before making a direct numerical
    comparison.
    """

    arr = np.asarray(
        img
    ).astype(
        np.float32
    )

    blur_radius = scale / 30.0

    blurred_img = img.filter(
        ImageFilter.GaussianBlur(
            radius=blur_radius
        )
    )

    blurred = np.asarray(
        blurred_img
    ).astype(
        np.float32
    )

    result = (
        arr * BEN_GRAHAM_ALPHA
        - blurred * BEN_GRAHAM_ALPHA
        + BEN_GRAHAM_BETA
    )

    result = np.clip(
        result,
        0,
        255,
    ).astype(
        np.uint8
    )

    return Image.fromarray(
        result
    )


# =============================================================================
# 22. MANIFEST
# =============================================================================

manifest_rows = []


# =============================================================================
# 23. WRITE VALIDATION + TEST
# =============================================================================
#
# These remain untouched.
#
# NO:
#   - augmentation
#   - Ben Graham
#   - geometric transformation
#
# This is critical.
# =============================================================================

log("\n" + "=" * 80)
log("WRITING VALIDATION AND TEST SETS")
log("=" * 80)

val_test_count = 0


for sample in samples:

    split = sample["split"]

    if split not in (
        "val",
        "test",
    ):
        continue

    source_path = sample["path"]

    destination_path = os.path.join(
        OUTPUT_DIR,
        split,
        sample["class"],
        sample["filename"],
    )

    shutil.copy2(
        source_path,
        destination_path,
    )

    manifest_rows.append(
        {
            "source_filename": sample["filename"],
            "source_path": sample["path"],
            "class": sample["class"],
            "split": split,
            "group_id": sample["group_id"],
            "derivation": "original_untouched",
            "output_filename": sample["filename"],
            "source_md5": sample["source_md5"],
        }
    )

    val_test_count += 1


log(
    f"Validation/test original images written: "
    f"{val_test_count}"
)


# =============================================================================
# 24. WRITE TRAINING SET
# =============================================================================
#
# TRAIN ONLY:
#
#   original
#   augmented
#   Ben Graham(original)
#
# If your ORIGINAL M4 pipeline used a different composition, change this
# section to exactly match it.
#
# Most importantly:
#
#   NO VALIDATION OR TEST IMAGE ENTERS THIS SECTION.
# =============================================================================

log("\n" + "=" * 80)
log("WRITING TRAINING SET")
log("=" * 80)


train_source_count = 0
train_output_count = 0


for sample in samples:

    if sample["split"] != "train":
        continue

    source_path = sample["path"]

    cls = sample["class"]

    original_filename = sample["filename"]

    stem, ext = os.path.splitext(
        original_filename
    )


    # -------------------------------------------------------------------------
    # LOAD SOURCE IMAGE
    # -------------------------------------------------------------------------

    try:

        orig_img = Image.open(
            source_path
        ).convert(
            "RGB"
        )

    except Exception as e:

        raise RuntimeError(
            f"Could not open image:\n"
            f"{source_path}\n"
            f"Error: {e}"
        )


    train_source_count += 1


    # -------------------------------------------------------------------------
    # 24.1 ORIGINAL TRAINING IMAGE
    # -------------------------------------------------------------------------

    original_output_filename = (
        f"orig__{stem}{ext}"
    )

    original_destination = os.path.join(
        OUTPUT_DIR,
        "train",
        cls,
        original_output_filename,
    )

    save_image(
        orig_img,
        original_destination,
    )

    manifest_rows.append(
        {
            "source_filename": original_filename,
            "source_path": source_path,
            "class": cls,
            "split": "train",
            "group_id": sample["group_id"],
            "derivation": "original",
            "output_filename": original_output_filename,
            "source_md5": sample["source_md5"],
        }
    )

    train_output_count += 1


    # -------------------------------------------------------------------------
    # 24.2 AUGMENTED TRAINING IMAGES
    # -------------------------------------------------------------------------

    for aug_index in range(
        AUGMENTATIONS_PER_IMAGE
    ):

        aug_img = augment_image(
            orig_img
        )

        aug_output_filename = (
            f"aug{aug_index}__{stem}{ext}"
        )

        aug_destination = os.path.join(
            OUTPUT_DIR,
            "train",
            cls,
            aug_output_filename,
        )

        save_image(
            aug_img,
            aug_destination,
        )

        manifest_rows.append(
            {
                "source_filename": original_filename,
                "source_path": source_path,
                "class": cls,
                "split": "train",
                "group_id": sample["group_id"],
                "derivation": f"augmented_{aug_index}",
                "output_filename": aug_output_filename,
                "source_md5": sample["source_md5"],
            }
        )

        train_output_count += 1


    # -------------------------------------------------------------------------
    # 24.3 BEN GRAHAM OF ORIGINAL TRAINING IMAGE
    # -------------------------------------------------------------------------

    bg_original = ben_graham(
        orig_img
    )

    bg_output_filename = (
        f"beng__{stem}{ext}"
    )

    bg_destination = os.path.join(
        OUTPUT_DIR,
        "train",
        cls,
        bg_output_filename,
    )

    save_image(
        bg_original,
        bg_destination,
    )

    manifest_rows.append(
        {
            "source_filename": original_filename,
            "source_path": source_path,
            "class": cls,
            "split": "train",
            "group_id": sample["group_id"],
            "derivation": "ben_graham_original",
            "output_filename": bg_output_filename,
            "source_md5": sample["source_md5"],
        }
    )

    train_output_count += 1


    # -------------------------------------------------------------------------
    # PROGRESS
    # -------------------------------------------------------------------------

    if train_source_count % 500 == 0:

        log(
            f"Processed {train_source_count} "
            f"training source images; "
            f"{train_output_count} training files written."
        )


log(
    f"\nTraining source images: "
    f"{train_source_count}"
)

log(
    f"Training files written: "
    f"{train_output_count}"
)


# =============================================================================
# 25. FINAL OUTPUT-LEVEL LEAKAGE CHECK
# =============================================================================
#
# Every derived training image must belong to a source image assigned to TRAIN.
#
# Validation/test must contain only "original_untouched".
# =============================================================================

log("\n" + "=" * 80)
log("FINAL MANIFEST LEAKAGE AUDIT")
log("=" * 80)


manifest_group_splits = defaultdict(set)

for row in manifest_rows:

    manifest_group_splits[
        row["group_id"]
    ].add(
        row["split"]
    )


for group_id, splits in manifest_group_splits.items():

    if len(splits) != 1:

        raise RuntimeError(
            "LEAKAGE DETECTED IN MANIFEST:\n"
            f"Group {group_id} appears in {splits}"
        )


# Check val/test derivations.

for row in manifest_rows:

    if row["split"] in (
        "val",
        "test",
    ):

        if row["derivation"] != "original_untouched":

            raise RuntimeError(
                "VALIDATION/TEST TRANSFORMATION ERROR:\n"
                f"{row}"
            )


# Check that training-derived images only originate from training sources.

for row in manifest_rows:

    if row["derivation"] != "original_untouched":

        if row["split"] != "train":

            raise RuntimeError(
                "NON-TRAIN DERIVATIVE DETECTED:\n"
                f"{row}"
            )


log(
    "✓ Every source-image group occurs in exactly one split."
)

log(
    "✓ Validation and test contain untouched originals only."
)

log(
    "✓ Augmented/Ben-Graham derivatives occur only in training."
)


# =============================================================================
# 26. WRITE MANIFEST
# =============================================================================

manifest_path = os.path.join(
    OUTPUT_DIR,
    "split_manifest.csv"
)


manifest_fields = [
    "source_filename",
    "source_path",
    "class",
    "split",
    "group_id",
    "derivation",
    "output_filename",
    "source_md5",
]


with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=manifest_fields,
    )

    writer.writeheader()

    writer.writerows(
        manifest_rows
    )


log(
    f"\nManifest saved:\n{manifest_path}"
)


# =============================================================================
# 27. WRITE CLASS DISTRIBUTION CSV
# =============================================================================

class_distribution_path = os.path.join(
    OUTPUT_DIR,
    "class_distribution.csv"
)


distribution_rows = []


for cls in classes:

    train_n = split_class_counts[
        "train"
    ][cls]

    val_n = split_class_counts[
        "val"
    ][cls]

    test_n = split_class_counts[
        "test"
    ][cls]

    total_n = (
        train_n
        + val_n
        + test_n
    )

    distribution_rows.append(
        {
            "class": cls,
            "train_source_images": train_n,
            "validation_source_images": val_n,
            "test_source_images": test_n,
            "total_source_images": total_n,
        }
    )


with open(
    class_distribution_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class",
            "train_source_images",
            "validation_source_images",
            "test_source_images",
            "total_source_images",
        ],
    )

    writer.writeheader()

    writer.writerows(
        distribution_rows
    )


# =============================================================================
# 28. WRITE SUMMARY
# =============================================================================

summary_path = os.path.join(
    OUTPUT_DIR,
    "split_summary.txt"
)


summary_lines = []


summary_lines.append(
    "OralVision M4 — Corrected Leakage-Aware Splitting Protocol"
)

summary_lines.append(
    "=" * 75
)

summary_lines.append(
    ""
)

summary_lines.append(
    f"Generated: {datetime.now().isoformat()}"
)

summary_lines.append(
    f"Random seed: {SEED}"
)

summary_lines.append(
    f"Input directory: {ORIGINAL_DIR}"
)

summary_lines.append(
    f"Output directory: {OUTPUT_DIR}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "SOURCE DATA"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    f"Original source images: {len(samples)}"
)

summary_lines.append(
    f"Number of classes: {len(classes)}"
)

summary_lines.append(
    f"Classes: {', '.join(classes)}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "SOURCE-IMAGE-LEVEL SPLIT"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    f"Train source images: {len(train_set)}"
)

summary_lines.append(
    f"Validation source images: {len(val_set)}"
)

summary_lines.append(
    f"Test source images: {len(test_set)}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "IMPORTANT GROUPING LIMITATION"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    "Grouping level: SOURCE IMAGE."
)

summary_lines.append(
    "Each original photograph is treated as one source-image group."
)

summary_lines.append(
    "No confirmed patient identifier was available to this script."
)

summary_lines.append(
    "Therefore TRUE patient-level separation has NOT been established."
)

summary_lines.append(
    ""
)

summary_lines.append(
    "PREPROCESSING"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    "Validation: original images only; no augmentation; no Ben Graham."
)

summary_lines.append(
    "Test: original images only; no augmentation; no Ben Graham."
)

summary_lines.append(
    "Training: original + augmented + Ben-Graham original."
)

summary_lines.append(
    ""
)

summary_lines.append(
    "AUGMENTATION SETTINGS"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    f"Augmented copies/source image: "
    f"{AUGMENTATIONS_PER_IMAGE}"
)

summary_lines.append(
    f"Rotation range: ±{ROTATION_RANGE_DEG} degrees"
)

summary_lines.append(
    f"Horizontal flip probability: "
    f"{HORIZONTAL_FLIP_PROBABILITY}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "LEAKAGE CHECKS"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    "Train/validation group overlap: NONE"
)

summary_lines.append(
    "Train/test group overlap: NONE"
)

summary_lines.append(
    "Validation/test group overlap: NONE"
)

summary_lines.append(
    "Validation/test transformed images: NONE"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "CLASS DISTRIBUTION"
)

summary_lines.append(
    "-" * 75
)


for cls in classes:

    summary_lines.append(
        f"{cls}: "
        f"train={split_class_counts['train'][cls]}, "
        f"val={split_class_counts['val'][cls]}, "
        f"test={split_class_counts['test'][cls]}"
    )


summary_lines.append(
    ""
)

summary_lines.append(
    "FILES"
)

summary_lines.append(
    "-" * 75
)

summary_lines.append(
    f"Manifest: {manifest_path}"
)

summary_lines.append(
    f"Class distribution: {class_distribution_path}"
)

summary_lines.append(
    f"Duplicate report: {duplicate_report_path}"
)


summary_text = "\n".join(
    summary_lines
)


with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        summary_text
    )


# =============================================================================
# 29. FINAL CONSOLE REPORT
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "ORALVISION M4 — CORRECTED SPLIT COMPLETE"
)

print(
    "=" * 80
)

print(
    f"\nOutput directory:\n{OUTPUT_DIR}"
)

print(
    f"\nOriginal source images: {len(samples)}"
)

print(
    f"Train source images:     {len(train_set)}"
)

print(
    f"Validation source images:{len(val_set)}"
)

print(
    f"Test source images:      {len(test_set)}"
)

print(
    f"\nTraining files written:   {train_output_count}"
)

print(
    f"Validation files written: {sum(split_class_counts['val'].values())}"
)

print(
    f"Test files written:       {sum(split_class_counts['test'].values())}"
)

print(
    "\nLEAKAGE AUDIT:"
)

print(
    "  ✓ Train / validation groups disjoint"
)

print(
    "  ✓ Train / test groups disjoint"
)

print(
    "  ✓ Validation / test groups disjoint"
)

print(
    "  ✓ Augmentation performed only after splitting"
)

print(
    "  ✓ Validation images untouched"
)

print(
    "  ✓ Test images untouched"
)

print(
    "\nIMPORTANT:"
)

print(
    "  This establishes SOURCE-IMAGE-LEVEL separation."
)

print(
    "  It does NOT establish PATIENT-LEVEL separation."
)

print(
    "  Patient-level separation requires confirmed patient/case IDs."
)

print(
    "\nGenerated files:"
)

print(
    f"  {manifest_path}"
)

print(
    f"  {class_distribution_path}"
)

print(
    f"  {duplicate_report_path}"
)

print(
    f"  {summary_path}"
)

print(
    "\nNext step:"
)

print(
    "  DO NOT modify train/val/test after this point."
)

print(
    "  Point the M4 training script to these three directories."
)

print(
    "  Train the corrected model and compare against the original protocol."
)

print(
    "\n" + "=" * 80
)

[17:25:56] ================================================================================
[17:25:56] ORALVISION M4 — CORRECTED SPLITTING PIPELINE
[17:25:56] ================================================================================
[17:25:56] Loading ORIGINAL source images...
[17:25:56] Input directory: C:\Users\R Y Z E N\Desktop\oral_merged
[17:25:56] Classes found: ['Calculus', 'Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
[17:25:56] Total original images: 15866
[17:25:56] 
Original class distribution:
[17:25:56]   Calculus                     2592
[17:25:56]   Caries                       2820
[17:25:56]   Gingivitis                   4698
[17:25:56]   Hypodontia                   2502
[17:25:56]   Mouth_Ulcer                  3071
[17:25:56]   Tooth_Discoloration           183
[17:25:56] 
[17:25:56] DIAGNOSTIC: FILENAME STRUCTURE
[17:25:56] ================================================================================
[17:25:56] 
Calculus — f

In [1]:
# =============================================================================
# OralVision M4
# CLEAN + DEDUPLICATED + LEAKAGE-AWARE DATASET PIPELINE
# =============================================================================
#
# PURPOSE
# -------
# 1. Start from ORIGINAL, pre-augmentation images.
# 2. Compute exact MD5 hashes.
# 3. Group pixel-identical images.
# 4. Keep ONE canonical image for same-label duplicate groups.
# 5. EXCLUDE duplicate groups containing conflicting labels.
# 6. Split CLEAN UNIQUE images before augmentation.
# 7. Keep validation/test completely untouched.
# 8. Apply augmentation + Ben Graham ONLY to training.
# 9. Produce a manifest and multiple leakage audits.
#
# IMPORTANT
# ---------
# This establishes IMAGE / EXACT-DUPLICATE GROUP separation.
# It does NOT establish PATIENT-LEVEL separation unless genuine patient/case/
# visit identifiers are available.
#
# =============================================================================

import os
import csv
import random
import hashlib
import shutil
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
from PIL import Image, ImageOps, ImageFilter
from sklearn.model_selection import train_test_split


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# ---------------------------------------------------------------------------
# INPUT: ORIGINAL DATASET ONLY
# ---------------------------------------------------------------------------
# Expected:
#
# oral_merged/
#     Calculus/
#     Caries/
#     Gingivitis/
#     Hypodontia/
#     Mouth_Ulcer/
#     Tooth_Discoloration/
#
# IMPORTANT:
# There must be NO augmented images already inside these folders.
# ---------------------------------------------------------------------------

ORIGINAL_DIR = r"C:\Users\R Y Z E N\Desktop\oral_merged"


# ---------------------------------------------------------------------------
# OUTPUT
# ---------------------------------------------------------------------------

OUTPUT_DIR = r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN"


# ---------------------------------------------------------------------------
# RANDOM SEED
# ---------------------------------------------------------------------------

SEED = 42


# ---------------------------------------------------------------------------
# SPLIT
# ---------------------------------------------------------------------------

TEST_SPLIT = 0.15
VAL_SPLIT = 0.15


# ---------------------------------------------------------------------------
# CONFLICT POLICY
# ---------------------------------------------------------------------------
#
# "exclude" is recommended for a single-label Q1 study.
#
# If an exact-identical image exists under multiple classes, the whole
# duplicate group is excluded unless you have a clinically defensible
# adjudication procedure.
#
# ---------------------------------------------------------------------------

CONFLICT_POLICY = "exclude"


# ---------------------------------------------------------------------------
# TRAINING AUGMENTATION
# ---------------------------------------------------------------------------

AUGMENTATIONS_PER_IMAGE = 1
ROTATION_RANGE_DEG = 15
HORIZONTAL_FLIP_PROBABILITY = 0.50


# ---------------------------------------------------------------------------
# BEN GRAHAM PARAMETERS
# ---------------------------------------------------------------------------
#
# IMPORTANT:
# Verify these exactly match your original M4 preprocessing.
#
# ---------------------------------------------------------------------------

BEN_GRAHAM_SCALE = 300
BEN_GRAHAM_ALPHA = 4.0
BEN_GRAHAM_BETA = 128.0


# ---------------------------------------------------------------------------
# VALID IMAGE EXTENSIONS
# ---------------------------------------------------------------------------

VALID_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
)


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================

random.seed(SEED)
np.random.seed(SEED)


# =============================================================================
# 3. LOGGING
# =============================================================================

def log(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


# =============================================================================
# 4. DIRECTORY VALIDATION
# =============================================================================

if not os.path.isdir(ORIGINAL_DIR):
    raise FileNotFoundError(
        f"\nOriginal dataset not found:\n{ORIGINAL_DIR}"
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 5. LOAD ORIGINAL DATASET
# =============================================================================

def build_samples(data_dir):
    classes = sorted(
        [
            d
            for d in os.listdir(data_dir)
            if os.path.isdir(os.path.join(data_dir, d))
        ]
    )

    if len(classes) < 2:
        raise RuntimeError(
            "Fewer than two class folders were found."
        )

    samples = []

    for cls in classes:

        class_dir = os.path.join(data_dir, cls)

        for fname in sorted(os.listdir(class_dir)):

            if not fname.lower().endswith(VALID_EXTENSIONS):
                continue

            full_path = os.path.join(class_dir, fname)

            samples.append(
                {
                    "path": full_path,
                    "class": cls,
                    "filename": fname,
                }
            )

    return samples, classes


# =============================================================================
# 6. MD5 HASH
# =============================================================================

def file_md5(path):

    md5 = hashlib.md5()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


# =============================================================================
# 7. BEN GRAHAM
# =============================================================================

def ben_graham(
    img,
    scale=BEN_GRAHAM_SCALE,
):

    arr = np.asarray(
        img
    ).astype(
        np.float32
    )

    blur_radius = scale / 30.0

    blurred_img = img.filter(
        ImageFilter.GaussianBlur(
            radius=blur_radius
        )
    )

    blurred = np.asarray(
        blurred_img
    ).astype(
        np.float32
    )

    result = (
        arr * BEN_GRAHAM_ALPHA
        - blurred * BEN_GRAHAM_ALPHA
        + BEN_GRAHAM_BETA
    )

    result = np.clip(
        result,
        0,
        255,
    ).astype(
        np.uint8
    )

    return Image.fromarray(result)


# =============================================================================
# 8. TRAINING AUGMENTATION
# =============================================================================

def augment_image(img):

    aug = img.copy()

    # Horizontal flip
    if random.random() < HORIZONTAL_FLIP_PROBABILITY:

        aug = ImageOps.mirror(
            aug
        )

    # Random rotation
    angle = random.uniform(
        -ROTATION_RANGE_DEG,
        ROTATION_RANGE_DEG,
    )

    aug = aug.rotate(
        angle,
        expand=False,
        fillcolor=(0, 0, 0),
    )

    return aug


# =============================================================================
# 9. SAVE IMAGE
# =============================================================================

def save_image(img, destination_path):

    ext = os.path.splitext(
        destination_path
    )[1].lower()

    if ext in (".jpg", ".jpeg"):

        img.save(
            destination_path,
            quality=95,
            optimize=True,
        )

    else:

        img.save(
            destination_path
        )


# =============================================================================
# 10. LOAD SOURCE IMAGES
# =============================================================================

log("=" * 90)
log("ORALVISION M4 — FINAL CLEAN DATASET PIPELINE")
log("=" * 90)

log("Loading ORIGINAL source images...")
log(f"Input directory: {ORIGINAL_DIR}")

samples, classes = build_samples(
    ORIGINAL_DIR
)

log(f"Classes: {classes}")
log(f"Total original files: {len(samples)}")


if len(samples) == 0:
    raise RuntimeError(
        "No images were found."
    )


# =============================================================================
# 11. ORIGINAL DISTRIBUTION
# =============================================================================

original_counts = Counter(
    s["class"]
    for s in samples
)

log("\nOriginal class distribution:")

for cls in classes:

    log(
        f"  {cls:<25} "
        f"{original_counts[cls]:>7}"
    )


# =============================================================================
# 12. COMPUTE EXACT HASHES
# =============================================================================

log("\n" + "=" * 90)
log("COMPUTING EXACT MD5 HASHES")
log("=" * 90)

hash_to_samples = defaultdict(list)

for i, sample in enumerate(samples, start=1):

    h = file_md5(
        sample["path"]
    )

    sample["md5"] = h

    hash_to_samples[h].append(
        sample
    )

    if i % 500 == 0:
        log(
            f"Hashed {i}/{len(samples)} files"
        )


log(
    f"Unique exact image hashes: "
    f"{len(hash_to_samples)}"
)


# =============================================================================
# 13. ANALYZE DUPLICATE GROUPS
# =============================================================================

log("\n" + "=" * 90)
log("ANALYZING EXACT-DUPLICATE GROUPS")
log("=" * 90)

same_label_groups = []
conflicting_groups = []

for h, group in hash_to_samples.items():

    labels = sorted(
        set(
            x["class"]
            for x in group
        )
    )

    if len(labels) == 1:

        if len(group) > 1:
            same_label_groups.append(
                {
                    "md5": h,
                    "count": len(group),
                    "class": labels[0],
                    "samples": group,
                }
            )

    else:

        conflicting_groups.append(
            {
                "md5": h,
                "count": len(group),
                "classes": labels,
                "samples": group,
            }
        )


duplicate_file_count = sum(
    len(x["samples"])
    for x in same_label_groups
)

conflict_file_count = sum(
    len(x["samples"])
    for x in conflicting_groups
)

log(
    f"Same-label duplicate groups: "
    f"{len(same_label_groups)}"
)

log(
    f"Files inside same-label duplicate groups: "
    f"{duplicate_file_count}"
)

log(
    f"Conflicting-label duplicate groups: "
    f"{len(conflicting_groups)}"
)

log(
    f"Files inside conflicting groups: "
    f"{conflict_file_count}"
)


# =============================================================================
# 14. SAVE DUPLICATE ANALYSIS
# =============================================================================

duplicate_analysis_path = os.path.join(
    OUTPUT_DIR,
    "duplicate_analysis.csv"
)

with open(
    duplicate_analysis_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "md5",
            "group_type",
            "group_size",
            "classes",
            "filename",
            "path",
        ],
    )

    writer.writeheader()

    # Same-label duplicates
    for group in same_label_groups:

        for sample in group["samples"]:

            writer.writerow(
                {
                    "md5": group["md5"],
                    "group_type": "same_label_duplicate",
                    "group_size": group["count"],
                    "classes": group["class"],
                    "filename": sample["filename"],
                    "path": sample["path"],
                }
            )

    # Conflicting groups
    for group in conflicting_groups:

        for sample in group["samples"]:

            writer.writerow(
                {
                    "md5": group["md5"],
                    "group_type": "conflicting_labels",
                    "group_size": group["count"],
                    "classes": "|".join(
                        group["classes"]
                    ),
                    "filename": sample["filename"],
                    "path": sample["path"],
                }
            )


# =============================================================================
# 15. SAVE CONFLICT REPORT
# =============================================================================

conflict_report_path = os.path.join(
    OUTPUT_DIR,
    "conflicting_duplicate_groups.csv"
)

with open(
    conflict_report_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "md5",
            "group_size",
            "classes",
            "filenames",
            "paths",
        ],
    )

    writer.writeheader()

    for group in conflicting_groups:

        writer.writerow(
            {
                "md5": group["md5"],
                "group_size": group["count"],
                "classes": "|".join(
                    group["classes"]
                ),
                "filenames": "|".join(
                    x["filename"]
                    for x in group["samples"]
                ),
                "paths": "|".join(
                    x["path"]
                    for x in group["samples"]
                ),
            }
        )


# =============================================================================
# 16. BUILD CLEAN UNIQUE DATASET
# =============================================================================
#
# Policy:
#
# SAME LABEL:
#     Keep one canonical image.
#
# CONFLICTING LABELS:
#     Exclude entire exact-duplicate group.
#
# =============================================================================

log("\n" + "=" * 90)
log("BUILDING CLEAN UNIQUE MASTER DATASET")
log("=" * 90)

clean_records = []
excluded_records = []

for h, group in hash_to_samples.items():

    labels = sorted(
        set(
            x["class"]
            for x in group
        )
    )

    # ------------------------------------------------------------
    # Conflict group
    # ------------------------------------------------------------

    if len(labels) > 1:

        if CONFLICT_POLICY == "exclude":

            for sample in group:

                excluded_records.append(
                    {
                        "reason": "conflicting_duplicate_labels",
                        "md5": h,
                        "class": sample["class"],
                        "filename": sample["filename"],
                        "path": sample["path"],
                    }
                )

            continue

        else:

            raise RuntimeError(
                "Unknown CONFLICT_POLICY."
            )

    # ------------------------------------------------------------
    # Same-label group
    # ------------------------------------------------------------

    # Deterministic canonical choice:
    # first file in sorted path order.
    canonical = sorted(
        group,
        key=lambda x: x["path"].lower()
    )[0]

    clean_records.append(
        {
            "md5": h,
            "class": canonical["class"],
            "filename": canonical["filename"],
            "path": canonical["path"],
            "duplicate_group_size": len(group),
            "source_duplicates_removed": max(
                0,
                len(group) - 1
            ),
        }
    )


log(
    f"Clean unique records: "
    f"{len(clean_records)}"
)

log(
    f"Excluded records due to conflicting labels: "
    f"{len(excluded_records)}"
)


# =============================================================================
# 17. SAVE EXCLUDED CONFLICT FILES
# =============================================================================

excluded_path = os.path.join(
    OUTPUT_DIR,
    "excluded_conflicting_images.csv"
)

with open(
    excluded_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "reason",
            "md5",
            "class",
            "filename",
            "path",
        ],
    )

    writer.writeheader()

    writer.writerows(
        excluded_records
    )


# =============================================================================
# 18. VERIFY CLEAN DATASET
# =============================================================================

clean_hashes = [
    x["md5"]
    for x in clean_records
]

if len(clean_hashes) != len(set(clean_hashes)):

    raise RuntimeError(
        "ERROR: duplicate hashes remain in clean dataset."
    )


clean_counts = Counter(
    x["class"]
    for x in clean_records
)

log("\nClean unique class distribution:")

for cls in classes:

    log(
        f"  {cls:<25} "
        f"{clean_counts[cls]:>7}"
    )


# =============================================================================
# 19. COPY CANONICAL CLEAN MASTER DATASET
# =============================================================================

MASTER_DIR = os.path.join(
    OUTPUT_DIR,
    "clean_master"
)

os.makedirs(
    MASTER_DIR,
    exist_ok=True
)

for cls in classes:

    os.makedirs(
        os.path.join(
            MASTER_DIR,
            cls
        ),
        exist_ok=True
    )


log("\nCopying canonical unique images...")

for i, record in enumerate(
    clean_records,
    start=1
):

    destination = os.path.join(
        MASTER_DIR,
        record["class"],
        record["filename"],
    )

    # Prevent name collision in the extremely unlikely case
    # that two canonical records share class + filename.
    if os.path.exists(destination):

        stem, ext = os.path.splitext(
            record["filename"]
        )

        destination = os.path.join(
            MASTER_DIR,
            record["class"],
            f"{stem}__{record['md5'][:8]}{ext}"
        )

    shutil.copy2(
        record["path"],
        destination
    )

    record["clean_master_path"] = destination

    if i % 500 == 0:
        log(
            f"Copied {i}/{len(clean_records)}"
        )


# =============================================================================
# 20. FINAL SPLIT
# =============================================================================
#
# Each exact-image hash is unique at this point.
# Split BEFORE augmentation.
#
# =============================================================================

log("\n" + "=" * 90)
log("CREATING FINAL CLEAN TRAIN / VALIDATION / TEST SPLIT")
log("=" * 90)

group_ids = [
    x["md5"]
    for x in clean_records
]

group_labels = [
    x["class"]
    for x in clean_records
]


# First: train+validation vs test
trainval_ids, test_ids = train_test_split(
    group_ids,
    test_size=TEST_SPLIT,
    stratify=group_labels,
    random_state=SEED,
)

group_to_record = {
    x["md5"]: x
    for x in clean_records
}

trainval_labels = [
    group_to_record[h]["class"]
    for h in trainval_ids
]

val_fraction_within_trainval = (
    VAL_SPLIT / (1.0 - TEST_SPLIT)
)

train_ids, val_ids = train_test_split(
    trainval_ids,
    test_size=val_fraction_within_trainval,
    stratify=trainval_labels,
    random_state=SEED,
)


train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)


# =============================================================================
# 21. SPLIT INTEGRITY
# =============================================================================

if train_ids & val_ids:
    raise RuntimeError(
        "Train/validation overlap detected."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test overlap detected."
    )

if val_ids & test_ids:
    raise RuntimeError(
        "Validation/test overlap detected."
    )

if (
    train_ids
    | val_ids
    | test_ids
) != set(group_ids):

    raise RuntimeError(
        "Some clean records were lost from the split."
    )


log(
    f"Train unique images: {len(train_ids)}"
)

log(
    f"Validation unique images: {len(val_ids)}"
)

log(
    f"Test unique images: {len(test_ids)}"
)


# =============================================================================
# 22. CREATE OUTPUT SPLIT DIRECTORIES
# =============================================================================

SPLIT_DIR = os.path.join(
    OUTPUT_DIR,
    "dataset"
)

for split in (
    "train",
    "val",
    "test",
):

    for cls in classes:

        os.makedirs(
            os.path.join(
                SPLIT_DIR,
                split,
                cls,
            ),
            exist_ok=True
        )


# =============================================================================
# 23. MANIFEST
# =============================================================================

manifest_rows = []


# =============================================================================
# 24. COPY VALIDATION + TEST
# =============================================================================
#
# These are ORIGINAL, untouched images.
#
# NO augmentation
# NO Ben Graham
#
# =============================================================================

log("\nWriting untouched validation and test sets...")

for split, ids in (
    ("val", val_ids),
    ("test", test_ids),
):

    for h in sorted(ids):

        record = group_to_record[h]

        destination = os.path.join(
            SPLIT_DIR,
            split,
            record["class"],
            record["filename"],
        )

        shutil.copy2(
            record["clean_master_path"],
            destination
        )

        manifest_rows.append(
            {
                "md5": record["md5"],
                "class": record["class"],
                "split": split,
                "source_original": record["path"],
                "clean_master": record["clean_master_path"],
                "output_filename": record["filename"],
                "derivation": "original_untouched",
            }
        )


# =============================================================================
# 25. WRITE TRAINING SET
# =============================================================================
#
# Training contains:
#
# 1. original
# 2. augmented
# 3. Ben Graham(original)
#
# ONLY training gets transformations.
#
# =============================================================================

log("\nWriting training set...")

for h in sorted(train_ids):

    record = group_to_record[h]

    cls = record["class"]

    orig_img = Image.open(
        record["clean_master_path"]
    ).convert(
        "RGB"
    )

    stem, ext = os.path.splitext(
        record["filename"]
    )

    # ------------------------------------------------------------
    # ORIGINAL
    # ------------------------------------------------------------

    original_filename = (
        f"orig__{stem}{ext}"
    )

    original_destination = os.path.join(
        SPLIT_DIR,
        "train",
        cls,
        original_filename,
    )

    save_image(
        orig_img,
        original_destination,
    )

    manifest_rows.append(
        {
            "md5": h,
            "class": cls,
            "split": "train",
            "source_original": record["path"],
            "clean_master": record["clean_master_path"],
            "output_filename": original_filename,
            "derivation": "original",
        }
    )

    # ------------------------------------------------------------
    # AUGMENTATION
    # ------------------------------------------------------------

    for aug_index in range(
        AUGMENTATIONS_PER_IMAGE
    ):

        aug_img = augment_image(
            orig_img
        )

        aug_filename = (
            f"aug{aug_index}__{stem}{ext}"
        )

        aug_destination = os.path.join(
            SPLIT_DIR,
            "train",
            cls,
            aug_filename,
        )

        save_image(
            aug_img,
            aug_destination,
        )

        manifest_rows.append(
            {
                "md5": h,
                "class": cls,
                "split": "train",
                "source_original": record["path"],
                "clean_master": record["clean_master_path"],
                "output_filename": aug_filename,
                "derivation": f"augmented_{aug_index}",
            }
        )

    # ------------------------------------------------------------
    # BEN GRAHAM
    # ------------------------------------------------------------

    bg_img = ben_graham(
        orig_img
    )

    bg_filename = (
        f"beng__{stem}{ext}"
    )

    bg_destination = os.path.join(
        SPLIT_DIR,
        "train",
        cls,
        bg_filename,
    )

    save_image(
        bg_img,
        bg_destination,
    )

    manifest_rows.append(
        {
            "md5": h,
            "class": cls,
            "split": "train",
            "source_original": record["path"],
            "clean_master": record["clean_master_path"],
            "output_filename": bg_filename,
            "derivation": "ben_graham_original",
        }
    )


# =============================================================================
# 26. MANIFEST LEAKAGE AUDIT
# =============================================================================

log("\n" + "=" * 90)
log("FINAL LEAKAGE AUDIT")
log("=" * 90)

manifest_group_splits = defaultdict(set)

for row in manifest_rows:

    manifest_group_splits[
        row["md5"]
    ].add(
        row["split"]
    )


bad_groups = {
    h: splits
    for h, splits in manifest_group_splits.items()
    if len(splits) != 1
}

if bad_groups:

    raise RuntimeError(
        f"LEAKAGE DETECTED: {len(bad_groups)} "
        "source hash groups occur in multiple splits."
    )


log(
    "✓ No source hash group occurs in more than one split."
)


# =============================================================================
# 27. VALIDATION / TEST TRANSFORMATION AUDIT
# =============================================================================

for row in manifest_rows:

    if row["split"] in (
        "val",
        "test",
    ):

        if row["derivation"] != "original_untouched":

            raise RuntimeError(
                "Validation/test transformation detected."
            )


log(
    "✓ Validation contains untouched originals only."
)

log(
    "✓ Test contains untouched originals only."
)


# =============================================================================
# 28. OUTPUT-LEVEL EXACT-HASH AUDIT
# =============================================================================
#
# Re-read the actual final folders rather than trusting the manifest.
#
# =============================================================================

final_hash_to_splits = defaultdict(set)

for split in (
    "train",
    "val",
    "test",
):

    split_root = os.path.join(
        SPLIT_DIR,
        split,
    )

    for root, _, files in os.walk(
        split_root
    ):

        for fname in files:

            if not fname.lower().endswith(
                VALID_EXTENSIONS
            ):
                continue

            path = os.path.join(
                root,
                fname
            )

            h = file_md5(
                path
            )

            final_hash_to_splits[h].add(
                split
            )


cross_split_exact_duplicates = {
    h: splits
    for h, splits in final_hash_to_splits.items()
    if len(splits) > 1
}


if cross_split_exact_duplicates:

    raise RuntimeError(
        "FINAL OUTPUT LEAKAGE DETECTED:\n"
        f"{len(cross_split_exact_duplicates)} "
        "exact hashes occur across partitions."
    )


log(
    "✓ Final output audit: train/val/test exact duplicates = ZERO."
)


# =============================================================================
# 29. SPLIT CLASS DISTRIBUTION
# =============================================================================

split_class_counts = {
    "train": Counter(),
    "val": Counter(),
    "test": Counter(),
}

for h in train_ids:
    split_class_counts["train"][
        group_to_record[h]["class"]
    ] += 1

for h in val_ids:
    split_class_counts["val"][
        group_to_record[h]["class"]
    ] += 1

for h in test_ids:
    split_class_counts["test"][
        group_to_record[h]["class"]
    ] += 1


log("\nFinal CLEAN split distribution:")

for cls in classes:

    log(
        f"{cls:<25} "
        f"train={split_class_counts['train'][cls]:>6}  "
        f"val={split_class_counts['val'][cls]:>6}  "
        f"test={split_class_counts['test'][cls]:>6}"
    )


# =============================================================================
# 30. SAVE FINAL MANIFEST
# =============================================================================

manifest_path = os.path.join(
    OUTPUT_DIR,
    "final_split_manifest.csv"
)

with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "md5",
            "class",
            "split",
            "source_original",
            "clean_master",
            "output_filename",
            "derivation",
        ],
    )

    writer.writeheader()

    writer.writerows(
        manifest_rows
    )


# =============================================================================
# 31. SAVE CLEAN MASTER MANIFEST
# =============================================================================

clean_master_manifest_path = os.path.join(
    OUTPUT_DIR,
    "clean_master_manifest.csv"
)

with open(
    clean_master_manifest_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "md5",
            "class",
            "filename",
            "path",
            "duplicate_group_size",
            "source_duplicates_removed",
            "clean_master_path",
        ],
    )

    writer.writeheader()

    writer.writerows(
        clean_records
    )


# =============================================================================
# 32. SAVE CLASS DISTRIBUTION
# =============================================================================

distribution_path = os.path.join(
    OUTPUT_DIR,
    "final_class_distribution.csv"
)

with open(
    distribution_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class",
            "train_unique_images",
            "val_unique_images",
            "test_unique_images",
            "total_unique_images",
        ],
    )

    writer.writeheader()

    for cls in classes:

        train_n = split_class_counts[
            "train"
        ][cls]

        val_n = split_class_counts[
            "val"
        ][cls]

        test_n = split_class_counts[
            "test"
        ][cls]

        writer.writerow(
            {
                "class": cls,
                "train_unique_images": train_n,
                "val_unique_images": val_n,
                "test_unique_images": test_n,
                "total_unique_images": (
                    train_n + val_n + test_n
                ),
            }
        )


# =============================================================================
# 33. SAVE SUMMARY
# =============================================================================

summary_path = os.path.join(
    OUTPUT_DIR,
    "FINAL_DATASET_SUMMARY.txt"
)

summary_lines = []

summary_lines.append(
    "OralVision M4 — FINAL CLEAN DATASET"
)

summary_lines.append(
    "=" * 80
)

summary_lines.append(
    f"Generated: {datetime.now().isoformat()}"
)

summary_lines.append(
    f"Random seed: {SEED}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "SOURCE DATA"
)

summary_lines.append(
    "-" * 80
)

summary_lines.append(
    f"Original image files: {len(samples)}"
)

summary_lines.append(
    f"Unique exact image hashes: {len(hash_to_samples)}"
)

summary_lines.append(
    f"Clean unique images retained: {len(clean_records)}"
)

summary_lines.append(
    f"Images excluded for conflicting labels: {len(excluded_records)}"
)

summary_lines.append(
    f"Same-label duplicate groups: {len(same_label_groups)}"
)

summary_lines.append(
    f"Conflicting-label duplicate groups: {len(conflicting_groups)}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "FINAL SPLIT"
)

summary_lines.append(
    "-" * 80
)

summary_lines.append(
    f"Train unique source images: {len(train_ids)}"
)

summary_lines.append(
    f"Validation unique source images: {len(val_ids)}"
)

summary_lines.append(
    f"Test unique source images: {len(test_ids)}"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "PREPROCESSING"
)

summary_lines.append(
    "-" * 80
)

summary_lines.append(
    "Validation: original images only; no augmentation; no Ben Graham."
)

summary_lines.append(
    "Test: original images only; no augmentation; no Ben Graham."
)

summary_lines.append(
    "Training: original + augmentation + Ben Graham."
)

summary_lines.append(
    ""
)

summary_lines.append(
    "LEAKAGE AUDIT"
)

summary_lines.append(
    "-" * 80
)

summary_lines.append(
    "Train/validation exact duplicate groups: ZERO"
)

summary_lines.append(
    "Train/test exact duplicate groups: ZERO"
)

summary_lines.append(
    "Validation/test exact duplicate groups: ZERO"
)

summary_lines.append(
    ""
)

summary_lines.append(
    "PATIENT-LEVEL LIMITATION"
)

summary_lines.append(
    "-" * 80
)

summary_lines.append(
    "Patient-level independence was NOT established."
)

summary_lines.append(
    "No confirmed patient/case/visit identifiers were available."
)

summary_lines.append(
    ""
)

summary_lines.append(
    "FINAL CLASS DISTRIBUTION"
)

summary_lines.append(
    "-" * 80
)

for cls in classes:

    summary_lines.append(
        f"{cls}: "
        f"train={split_class_counts['train'][cls]}, "
        f"val={split_class_counts['val'][cls]}, "
        f"test={split_class_counts['test'][cls]}"
    )


with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(summary_lines)
    )


# =============================================================================
# 34. FINAL REPORT
# =============================================================================

log("\n" + "=" * 90)
log("FINAL CLEAN DATASET PIPELINE COMPLETE")
log("=" * 90)

log(
    f"Original files:                  {len(samples)}"
)

log(
    f"Unique exact images:             {len(hash_to_samples)}"
)

log(
    f"Clean unique images retained:    {len(clean_records)}"
)

log(
    f"Conflicting images excluded:     {len(excluded_records)}"
)

log(
    f"Train unique images:             {len(train_ids)}"
)

log(
    f"Validation unique images:        {len(val_ids)}"
)

log(
    f"Test unique images:              {len(test_ids)}"
)

log(
    "\nFINAL LEAKAGE AUDIT:"
)

log(
    "  ✓ No exact duplicate across train/validation"
)

log(
    "  ✓ No exact duplicate across train/test"
)

log(
    "  ✓ No exact duplicate across validation/test"
)

log(
    "  ✓ Validation untouched"
)

log(
    "  ✓ Test untouched"
)

log(
    "  ✓ Augmentation only in training"
)

log(
    "\nIMPORTANT:"
)

log(
    "  Patient-level independence is NOT established."
)

log(
    "\nOutput:"
)

log(
    f"  {OUTPUT_DIR}"
)

log(
    "\nFiles created:"
)

log(
    f"  {duplicate_analysis_path}"
)

log(
    f"  {conflict_report_path}"
)

log(
    f"  {excluded_path}"
)

log(
    f"  {clean_master_manifest_path}"
)

log(
    f"  {manifest_path}"
)

log(
    f"  {distribution_path}"
)

log(
    f"  {summary_path}"
)

log("=" * 90)

[17:57:07] ==========================================================================================
[17:57:07] ORALVISION M4 — FINAL CLEAN DATASET PIPELINE
[17:57:07] ==========================================================================================
[17:57:07] Loading ORIGINAL source images...
[17:57:07] Input directory: C:\Users\R Y Z E N\Desktop\oral_merged
[17:57:08] Classes: ['Calculus', 'Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
[17:57:08] Total original files: 15866
[17:57:08] 
Original class distribution:
[17:57:08]   Calculus                     2592
[17:57:08]   Caries                       2820
[17:57:08]   Gingivitis                   4698
[17:57:08]   Hypodontia                   2502
[17:57:08]   Mouth_Ulcer                  3071
[17:57:08]   Tooth_Discoloration           183
[17:57:08] 
[17:57:08] COMPUTING EXACT MD5 HASHES
[17:57:08] ==========================================================================================
[17:5